# Verification workflow demo

Synthetic deterministic checks only. The output is not scientific validation, a financial value, or a researcher ranking.

In [1]:
import csv, hashlib, json, os
from pathlib import Path
from mct_reward_simulation import load_events, score_events, summary

INPUT_PATH = Path("data/example_contributions.json")
OUTPUT_DIR = Path(os.environ.get("MCT_OUTPUT_DIR", "outputs"))
events = load_events(INPUT_PATH)
assert len(events) == 6
assert {event["schema_version"] for event in events} == {"0.3.4-alpha"}
print(json.dumps({"event_count": len(events), "schema_version": "0.3.4-alpha"}, sort_keys=True))


{"event_count": 6, "schema_version": "0.3.4-alpha"}


In [2]:
rows = score_events(events, half_life_days=365.0)
run_summary = summary(rows, INPUT_PATH, 365.0)
assert run_summary["diagnostic_score_sum"] == 23.0324
assert run_summary["num_events"] == 6
print(json.dumps({"diagnostic_score_sum": run_summary["diagnostic_score_sum"], "row_count": len(rows)}, sort_keys=True))


{"diagnostic_score_sum": 23.0324, "row_count": 6}


In [3]:
verification_rows = []
for event in events:
    validation = event["validation"]
    verification_rows.append({
        "event_id": event["event_id"],
        "metadata_status": validation["metadata_status"],
        "evidence_file_status": validation["evidence_file_status"],
        "file_integrity_status": validation["file_integrity_status"],
        "source_link_status": validation["source_link_status"],
        "scientific_assessment_status": validation["scientific_assessment"]["status"],
        "credential_locked": event["issued_credential"]["locked"],
        "contributor_verifier_separated": event["contributor"]["orcid"] != validation["verifier"]["identifier"],
    })
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
verification_path = OUTPUT_DIR / "verification_results.csv"
with verification_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(verification_rows[0]))
    writer.writeheader()
    writer.writerows(verification_rows)
observed_states = {field: sorted({row[field] for row in verification_rows}) for field in ("metadata_status", "evidence_file_status", "file_integrity_status", "source_link_status", "scientific_assessment_status")}
print(json.dumps({"observed_validation_states": observed_states, "output": "verification_results.csv", "verification_results_sha256": hashlib.sha256(verification_path.read_bytes()).hexdigest(), "verification_row_count": len(verification_rows)}, sort_keys=True))


{"observed_validation_states": {"evidence_file_status": ["evidence_file_present", "not_checked"], "file_integrity_status": ["file_integrity_confirmed", "not_checked"], "metadata_status": ["metadata_validated"], "scientific_assessment_status": ["not_reviewed"], "source_link_status": ["source_link_recorded"]}, "output": "verification_results.csv", "verification_results_sha256": "5d245030b0d64878d8d2cb73754c1b66a417770d02bc694ffe148e6eec5171d9", "verification_row_count": 6}
